In [0]:
from pyspark.sql import functions as F

def add_ingestion_metadata(input_df):
  return (
    input_df
      .withColumn("ingestion_timestamp", F.current_timestamp())
      .withColumn("source_file", F.col('_metadata.file_path')))

In [0]:
def write_to_bronze (input_df, target_table, batch_id):
    import pyspark.sql.functions as f

    final_df = input_df.withColumn("batch_id", f.lit(batch_id))
    (
        final_df.write.format("delta")
        .mode("overwrite")
        .partitionBy("batch_id")
        .option("replaceWhere", f"batch_id='{batch_id}'")
        .saveAsTable(target_table)
    )

In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as f


def write_to_silver(input_df, target_table, merge_condition, columns_to_update):
    """
    creates the Delta table if it does not exist.
    otherwise merges the input Dataframe into the target table.
    """

    final_df = input_df.withColumn(
        "created_timestamp", f.current_timestamp()
    ).withColumn("updated_timestamp", f.current_timestamp())

    if not spark.catalog.tableExists(target_table):
        (final_df.write.format("delta").mode("overwrite").saveAsTable(target_table))
    else:
        delta_table = DeltaTable.forName(spark, target_table)

        update_map = {column: f"s.{column}" for column in columns_to_update}
        update_map["updated_timestamp"] = "s.updated_timestamp"

        (
            delta_table.alias("t")
            .merge(final_df.alias("s"), merge_condition)
            .whenMatchedUpdate(
                condition=f"s.batch_id >= t.batch_id",
                set=update_map)
            .whenNotMatchedInsertAll()
            .execute()
        )

In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as f


def write_to_gold(input_df, target_table, merge_condition, columns_to_update):
    """
    creates the Delta table if it does not exist.
    otherwise merges the input Dataframe into the target table.
    """

    final_df = input_df.withColumn(
        "created_timestamp", f.current_timestamp()
    ).withColumn("updated_timestamp", f.current_timestamp())

    if not spark.catalog.tableExists(target_table):
        (final_df.write.format("delta").mode("overwrite").saveAsTable(target_table))
    else:
        delta_table = DeltaTable.forName(spark, target_table)

        update_map = {column: f"s.{column}" for column in columns_to_update}
        update_map["updated_timestamp"] = "s.updated_timestamp"

        (
            delta_table.alias("t")
            .merge(final_df.alias("s"), merge_condition)
            .whenMatchedUpdate(
                set=update_map)
            .whenNotMatchedInsertAll()
            .execute()
        )